In [5]:
import pandas as pd
import os
import numpy as np

# читаем исходный датасет
df = pd.read_csv("train.csv")

# считаем количество строк на каждый nid
nid_counts = df["nid"].value_counts().reset_index()
nid_counts.columns = ["nid", "count"]

# сортируем по количеству строк
nid_counts = nid_counts.sort_values("count").reset_index(drop=True)

# задаем желаемое количество частей
num_parts = 5  # можно поменять

# считаем общее количество строк и целевой размер части
total_rows = nid_counts["count"].sum()
target_rows = total_rows / num_parts

# разбиваем по логике "равных сумм строк", но сохраняя близкие count
parts = []
current_part = []
current_sum = 0

for _, row in nid_counts.iterrows():
    nid, count = row["nid"], row["count"]
    if current_sum + count > target_rows and len(parts) < num_parts - 1:
        parts.append(current_part)
        current_part = [nid]
        current_sum = count
    else:
        current_part.append(nid)
        current_sum += count

# добавляем последний кусок
if current_part:
    parts.append(current_part)

# список с датасетами
datasets = []

for i, part in enumerate(parts, 1):
    part_df = df[df["nid"].isin(part)].copy()
    datasets.append(part_df)

    stats = part_df["nid"].value_counts()
    part_min = stats.min()
    part_max = stats.max()

    # считаем количество нулей
    week_zeros = (part_df["week_churn"] == 0).sum()
    month_zeros = (part_df["month_churn"] == 0).sum()

    print(f"\n=== part_{i} ===")
    print(f"Количество уникальных nid: {len(stats)}")
    print(f"Диапазон строк на nid: {part_min}–{part_max}")
    print(f"Всего строк: {len(part_df)}")
    print(f"Нулей в week_churn: {week_zeros} ({week_zeros / len(part_df) * 100:.2f}%)")
    print(f"Нулей в month_churn: {month_zeros} ({month_zeros / len(part_df) * 100:.2f}%)")

# теперь datasets — список из 10 DataFrame



=== part_1 ===
Количество уникальных nid: 6947
Диапазон строк на nid: 1–20
Всего строк: 49621
Нулей в week_churn: 26164 (52.73%)
Нулей в month_churn: 32052 (64.59%)

=== part_2 ===
Количество уникальных nid: 1933
Диапазон строк на nid: 20–32
Всего строк: 49636
Нулей в week_churn: 36781 (74.10%)
Нулей в month_churn: 44650 (89.95%)

=== part_3 ===
Количество уникальных nid: 1228
Диапазон строк на nid: 32–50
Всего строк: 49602
Нулей в week_churn: 37650 (75.90%)
Нулей в month_churn: 44488 (89.69%)

=== part_4 ===
Количество уникальных nid: 865
Диапазон строк на nid: 50–63
Всего строк: 49605
Нулей в week_churn: 40791 (82.23%)
Нулей в month_churn: 47042 (94.83%)

=== part_5 ===
Количество уникальных nid: 730
Диапазон строк на nid: 63–79
Всего строк: 49716
Нулей в week_churn: 45042 (90.60%)
Нулей в month_churn: 47795 (96.14%)


In [8]:
datasets[0]

,month_churn,nid,report_date,v_0,v_1,v_10,v_100,v_101,v_102,v_103,...,v_91,v_92,v_93,v_94,v_95,v_96,v_97,v_98,v_99,week_churn
2,0,9732,2023-10-29,35,39,19,19,6,6,0,...,0,0,45,76,1,0,19,19,19,1
3,1,2934,2024-12-08,11,15,101,91,31,48,0,...,0,1,122,460,0,0,93,93,84,1
6,0,4039,2024-11-17,119,233,65,62,21,21,0,...,0,0,120,234,0,0,65,65,48,0
9,0,650,2025-01-19,88,341,31,31,25,26,0,...,0,0,89,347,5,12,31,31,30,0
11,0,3756,2025-03-09,19,19,83,6,4,4,0,...,0,0,19,19,0,0,6,6,6,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248149,1,3413,2023-09-17,3,3,59,4,1,3,0,...,0,0,3,3,0,1,59,59,1,1
248159,0,2960,2024-12-08,17,27,71,69,17,21,0,...,0,0,27,59,2,2,71,71,36,0
248160,0,7336,2024-11-24,76,117,89,82,27,27,0,...,0,0,81,637,0,0,88,88,78,1
248169,0,6030,2024-09-29,8,9,14,5,2,5,0,...,0,0,8,9,0,1,6,6,2,0
